# 03 · Preprocessing for knowledge engineering
Reads the cleaned file and applies the functions of `scripts/preprocess_dataset.py`. Original engineering
units are preserved; only derived columns are added.

In [1]:
import sys, pandas as pd
from IPython.display import Image, Markdown, display
sys.path.insert(0, "../scripts")
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
import preprocess_dataset as pp
df, reg = pp.load()
reg.head()

,Pump_ID,Pump_Type,Manufacturer,Installation_Date,Rated_Flow_m3h,Rated_Differential_bar,Nominal_Suction_bar_a,Rated_Speed_RPM,Rated_Current_A,Utilisation
0,P-001,End Suction,Vendor C,2011-06-25,134.0,6.84,1.98,2950,54.0,0.72
1,P-002,Multistage,Vendor A,2020-10-03,110.0,14.85,1.99,2950,45.0,0.77
2,P-003,End Suction,Vendor A,2021-06-01,164.0,3.54,1.23,1475,82.0,0.83
3,P-004,End Suction,Vendor A,2013-08-25,183.0,4.15,1.37,1475,68.0,0.71
4,P-005,End Suction,Vendor D,2018-06-29,128.0,4.41,1.49,1475,59.0,0.70


In [2]:
df = pp.date_features(df)
df = pp.engineering_features(df, reg)
df[["Pump_ID", "Flow_Rate", "Rated_Flow_m3h", "Flow_Pct_Rated", "Head_Pct_Rated", "Current_Pct_Rated",
    "Speed_Deviation_Pct", "Vibration_Zone", "Pump_Age_Years", "Hours_Since_Last_Maintenance"]].head(8)

,Pump_ID,Flow_Rate,Rated_Flow_m3h,Flow_Pct_Rated,Head_Pct_Rated,Current_Pct_Rated,Speed_Deviation_Pct,Vibration_Zone,Pump_Age_Years,Hours_Since_Last_Maintenance
0,P-001,130.9,134.0,97.69,101.75,97.04,0.102,A,11.52,<NA>
1,P-001,114.0,134.0,85.07,106.29,87.22,0.068,A,11.53,<NA>
2,P-001,115.6,134.0,86.27,105.85,87.04,0.068,B,11.55,<NA>
3,P-001,122.2,134.0,91.19,103.36,91.3,0.068,B,11.56,<NA>
4,P-001,132.1,134.0,98.58,100.58,93.7,0.034,B,11.58,<NA>
5,P-001,129.4,134.0,96.57,106.43,99.63,-0.068,D,11.59,<NA>
6,P-001,126.1,134.0,94.1,103.36,97.96,-0.068,D,11.60,<NA>
7,P-001,121.4,134.0,90.6,103.8,92.41,0.068,B,11.62,122.0


## Sensor-derived symptoms (1/0/<NA>)
<NA> whenever an input was imputed — evidence is never derived from a guessed value.

In [3]:
df = pp.symptom_features(df)
pd.DataFrame([(k, ", ".join(c), j) for k, c, _, j in pp.THRESHOLDS], columns=["key", "source", "rule"])

,key,source,rule
0,vibration_high,Vibration_mm_s,> 4.5 mm/s RMS — upper limit of zone C in ISO ...
1,bearing_temp_high,Bearing_Temperature_C,≥ 80 °C — same threshold as the existing KBS (...
2,seal_temp_high,Seal_Temperature_C,"≥ 80 °C — prototype alarm level, ~30 °C above ..."
3,motor_temp_high,Motor_Temperature_C,≥ 95 °C frame temperature — prototype alarm level
4,flow_low,Flow_Rate,10–75 % of rated flow — same 75 % threshold as...
5,no_flow,Flow_Rate,< 10 % of rated flow — as in the existing KBS
6,discharge_pressure_low,"Discharge_Pressure, Suction_Pressure","pump differential < 85 % of rated (relative, s..."
7,discharge_pressure_high,"Discharge_Pressure, Suction_Pressure",pump differential > 115 % of rated — approachi...
8,suction_pressure_low,Suction_Pressure,"< 0.8 bar(a) — sub-atmospheric suction, cavita..."
9,motor_current_high,Motor_Current,> 110 % of rated current


In [4]:
df.filter(like='sens_').apply(lambda s: s.value_counts(dropna=False)).T

,0,1,<NA>
sens_vibration_high,5510,1552,148
sens_bearing_temp_high,6669,400,141
sens_seal_temp_high,6683,369,158
sens_motor_temp_high,6941,127,142
sens_flow_low,6355,714,141
sens_no_flow,6974,95,141
sens_discharge_pressure_low,6800,139,271
sens_discharge_pressure_high,6770,169,271
sens_suction_pressure_low,6723,350,137
sens_motor_current_high,6990,71,149


## Encodings and group-aware split

In [5]:
df = pp.encodings(df)
df = pp.partition(df)
display(df.groupby("Partition").Pump_ID.nunique())
pd.crosstab(df.Fault_Type, df.Partition)

Partition
test           8
train         24
validation     8
Name: Pump_ID, dtype: int64

Partition,test,train,validation
Fault_Type,,,
Air Ingress,26,108,28
Bearing Failure,116,412,139
Cavitation,61,152,36
Discharge Restriction,28,103,35
Dry Running,16,40,16
Impeller Damage,48,152,71
Mechanical Seal Leakage,124,343,148
Motor Fault,34,105,25
Normal,879,2612,856


Leakage check — no pump appears in more than one partition:

In [6]:
(df.groupby('Pump_ID').Partition.nunique() > 1).sum()

np.int64(0)

## Standardisation fitted on train only

In [7]:
df, params = pp.standardise(df)
pd.DataFrame(params).T

,mean,std
Vibration_mm_s,3.210288,2.527128
Bearing_Temperature_C,55.654335,10.154932
Seal_Temperature_C,51.564730,11.269825
Motor_Temperature_C,68.902776,7.129595
Flow_Pct_Rated,86.894551,16.349000
Head_Pct_Rated,102.112984,11.236431
Current_Pct_Rated,89.502138,9.443613
Speed_Deviation_Pct,-0.009553,0.357689
Pressure_Fluctuation,3.035127,3.815729
Noise_Level,78.627847,3.896505


## Final schema validation

In [8]:
val = pp.validate(df)
print('violations:', int(val.violations.sum()))
val.head(20)

violations: 0


,column,check,violations
0,Record_ID,no nulls,0
1,Record_ID,matches ^R-\d{6}$,0
2,Pump_ID,no nulls,0
3,Pump_ID,matches ^P-\d{3}$,0
4,Pump_Type,no nulls,0
5,Pump_Type,allowed values,0
6,Manufacturer,no nulls,0
7,Manufacturer,allowed values,0
8,Installation_Date,no nulls,0
9,Installation_Date,ISO date,0
